# Contrôle de cohérence : recensement (MOBPRO) et enquête mobilité (EMC²)

**But** : sur un territoire couvert par une enquête mobilité certifiée Cerema (EMC²), comparer la part de la voiture
pour aller travailler selon le recensement (fichier MOBPRO, celui de la carte) et selon l'enquête, zone par zone.
Si les deux sources donnent des ordres de grandeur proches, la carte est crédible sur ce point.

**Comment faire** : choisir l'enquête dans **Paramètres**, puis **Exécution → Tout exécuter** (5 à 10 minutes).

**Attention, deux mesures différentes**
- MOBPRO : le mode de transport **habituel** de chaque actif pour aller travailler (déclaré au recensement).
- EMC² : les **déplacements** pour motif travail réellement faits la veille de l'enquête, un jour de semaine.

On s'attend donc à des écarts de quelques points, pas à des chiffres identiques.

**Étape manuelle** : la mise en page des tableaux EMC² varie d'une enquête à l'autre. Le notebook trouve
les tableaux candidats et les affiche. Il faut ensuite recopier les parts de la voiture dans la cellule 6
(une ligne par zone).

## 1. Paramètres

In [ ]:
# Identifiant data.gouv.fr de l'exploitation standard (fin de l'adresse de la page du jeu de données)
EMC2_DATASET = "68da310010186f209b48f404"      # Grand Angoulême 2023 (Charente)
# Autres exemples : "695bc1a467c9c1f655573a45" Arras 2024 · "695b9e4a2e3fd5407c401148" Grand Amiénois 2024
#                   "68dd278b0389a6ef7f8415e7" Calvados 2022 · "6aa26cbdcefeb8375171bff8" Clermont-Ferrand 2023
#                   "68dd34047a4a5123b6c7a8b7" Tours 2019 · "6aa2675d5c327d7de302aec9" Nord Franche-Comté 2024
# Liste complète : https://www.data.gouv.fr/datasets/?q=EMC%C2%B2%20exploitation%20standard

NIVEAU_ZONE = None     # colonne de zonage à utiliser (ex. "D10", "D5", "D2") ; None = choix automatique
MOBPRO_URL = "https://www.insee.fr/fr/statistiques/fichier/8589904/RP2022_mobpro.parquet"
VARMOD_URL = "https://www.insee.fr/fr/statistiques/fichier/8589904/varmod_mobpro_2022.csv"

## 2. Installation et outils
Rien à modifier ici.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "py7zr", "pyarrow", "openpyxl", "geopandas"], check=False)

import re, time, zipfile
from pathlib import Path
import numpy as np, pandas as pd, requests

DATA = Path("emc2_brut"); DATA.mkdir(exist_ok=True)
UA = {"User-Agent": "Mozilla/5.0 (prototype dependance-auto; open data)"}
RE_COM = re.compile(r"^(\d{5}|2[AB]\d{3})$")

def download(url, dest):
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    with requests.get(url, headers=UA, stream=True, timeout=600) as r:
        r.raise_for_status()
        with open(dest, "wb") as f:
            for ch in r.iter_content(1 << 20):
                f.write(ch)
    print(f"  ✓ {dest.name} ({dest.stat().st_size/1e6:.1f} Mo)")
    return dest

def extraire(archive, dossier):
    dossier.mkdir(parents=True, exist_ok=True)
    if archive.suffix.lower() == ".7z":
        import py7zr
        with py7zr.SevenZipFile(archive) as z:
            z.extractall(dossier)
    elif archive.suffix.lower() == ".zip":
        zipfile.ZipFile(archive).extractall(dossier)
    # archives emboîtées
    for sub in list(dossier.rglob("*.7z")) + list(dossier.rglob("*.zip")):
        extraire(sub, sub.with_suffix(""))
        sub.unlink()

def part_codes_commune(s):
    s = s.dropna().astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
    s = s[s.str.len().isin([4, 5])].str.zfill(5).reindex(s.index)   # 4 chiffres : zéro initial perdu par le tableur
    return s.fillna("").str.match(RE_COM).mean() if len(s) else 0

## 3. Téléchargement de l'enquête

In [ ]:
ds = requests.get(f"https://www.data.gouv.fr/api/1/datasets/{EMC2_DATASET}/", headers=UA, timeout=60).json()
TITRE = ds["title"]
print(TITRE)
ROOT = DATA / EMC2_DATASET
for r in ds["resources"]:
    url, nom = r["url"], Path(r["url"].split("?")[0]).name
    if nom.lower().endswith((".7z", ".zip")):
        arch = download(url, DATA / f"{EMC2_DATASET}_{nom}")
        extraire(arch, ROOT / Path(nom).stem)
    elif nom.lower().endswith(".pdf"):
        download(url, ROOT / nom) if ROOT.exists() else None

fichiers = sorted(p for p in ROOT.rglob("*") if p.is_file())
print(f"\n{len(fichiers)} fichiers :")
for p in fichiers:
    print("  ", p.relative_to(ROOT), f"({p.stat().st_size/1e3:.0f} ko)")

## 4. Le sommaire des tableaux
La liste de tous les tableaux de l'exploitation standard, avec leur niveau géographique.

In [ ]:
pd.set_option("display.max_rows", 200, "display.max_colwidth", 120)
som = [p for p in fichiers if "sommaire" in p.name.lower() and p.suffix.lower() in (".xlsx", ".xls", ".csv")]
if som:
    s = pd.read_excel(som[0], header=None) if som[0].suffix.lower() != ".csv" else pd.read_csv(som[0], sep=None, engine="python")
    display(s.dropna(how="all").head(200))
else:
    print("Pas de fichier « sommaire » trouvé ; voir la liste des fichiers ci-dessus.")

## 5. Les communes de chaque zone
On cherche, dans les annexes (tableurs et fichiers SIG), une table qui donne pour chaque commune son code et sa zone.

In [ ]:
def lire_tables(p):
    """Toutes les tables d'un fichier, sous forme de DataFrames (en-tête cherché dans les 15 premières lignes)."""
    ext = p.suffix.lower()
    try:
        if ext in (".xlsx", ".xls"):
            out = []
            for sh, raw in pd.read_excel(p, sheet_name=None, header=None, dtype=str).items():
                for h in range(min(15, len(raw))):
                    row = raw.iloc[h].dropna().astype(str)
                    if len(row) >= 2 and row.str.match(r"^[A-Za-z_]").mean() > 0.7:
                        df = raw.iloc[h + 1:].copy(); df.columns = [str(c).strip() for c in raw.iloc[h]]
                        out.append((f"{p.name} › {sh}", df.dropna(how="all")))
                        break
            return out
        if ext in (".csv", ".txt"):
            return [(p.name, pd.read_csv(p, sep=None, engine="python", dtype=str, encoding_errors="replace"))]
        if ext in (".shp", ".dbf", ".gpkg", ".geojson", ".tab", ".mif"):
            import geopandas as gpd
            g = gpd.read_file(p if ext != ".dbf" else p.with_suffix(".shp") if p.with_suffix(".shp").exists() else p)
            return [(p.name, pd.DataFrame(g.drop(columns="geometry", errors="ignore")).astype(str))]
    except Exception as e:
        print("  (illisible)", p.name, "—", str(e)[:80])
    return []

cands = []
for p in fichiers:
    if p.suffix.lower() == ".dbf" and p.with_suffix(".shp").exists():
        continue
    for nom, df in lire_tables(p):
        if len(df) < 2:
            continue
        cc = max(df.columns, key=lambda c: part_codes_commune(df[c]))
        if part_codes_commune(df[cc]) < 0.8:
            continue
        zones = [c for c in df.columns if re.search(r"(^|[^A-Z])D(2|5|10|30)([^0-9]|$)|DTIR|ZONE|SECT", str(c).upper())
                 and c != cc]
        cands.append((nom, cc, zones, df))
        print(f"• {nom} : {len(df)} lignes, code commune = « {cc} », zonages = {zones}")

if not cands:
    raise SystemExit("Aucune table communes → zones trouvée : envoyez la liste des fichiers ci-dessus pour adapter le notebook.")
# on préfère une table avec des zonages et une ligne par commune
nom, cc, zones, df = max(cands, key=lambda t: (len(t[2]) > 0, -abs(t[3][t[1]].nunique() - len(t[3]))))
if NIVEAU_ZONE and NIVEAU_ZONE in df:
    zcol = NIVEAU_ZONE
else:
    pref = [z for k in ("D10", "D5", "D30", "D2") for z in zones if re.search(rf"(^|[^0-9]){k}([^0-9]|$)", z.upper())]
    zcol = (pref or zones or [None])[0]
print(f"\nTable retenue : {nom} — colonne commune « {cc} », zone « {zcol} »")
corr = pd.DataFrame({"commune": df[cc].astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(5),
                     "zone": df[zcol].astype(str) if zcol else "Ensemble"})
lib = [c for c in df.columns if zcol and str(c).upper().startswith(("LIB", "NOM")) and re.search(zcol[-2:], str(c))]
if lib:
    corr["zone_nom"] = df[lib[0]].values
corr = corr[corr["commune"].str.match(RE_COM)]
# une commune peut être découpée en plusieurs secteurs : on garde la zone la plus fréquente
corr = corr.groupby("commune").agg(lambda s: s.mode().iloc[0]).reset_index()
print(f"{len(corr)} communes, {corr['zone'].nunique()} zones")
display(corr.groupby("zone").size().rename("communes").to_frame())

## 6. La part de la voiture pour aller travailler selon le recensement (MOBPRO)

In [ ]:
mob = download(MOBPRO_URL, DATA / "RP2022_mobpro.parquet")
vm = download(VARMOD_URL, DATA / "varmod_mobpro_2022.csv")
v = pd.read_csv(vm, sep=None, engine="python", dtype=str, encoding_errors="replace")
v.columns = [c.upper() for c in v.columns]
t = v[v.iloc[:, 0].str.upper() == "TRANS"]
lab = dict(zip(t["COD_MOD"], t["LIB_MOD"]))
code_voit = [c for c, l in lab.items() if "voiture" in l.lower()]
code_aucun = [c for c, l in lab.items() if "pas de transport" in l.lower()]
print("Modes :", lab)

import pyarrow.dataset as pads
d = pads.dataset(mob)
cols = [c for c in d.schema.names if c.upper() in ("COMMUNE", "TRANS", "IPONDI")]
m = d.to_table(columns=cols, filter=pads.field(cols[[c.upper() for c in cols].index("COMMUNE")])
               .isin(corr["commune"].tolist())).to_pandas()
m.columns = [c.upper() for c in m.columns]
m["IPONDI"] = m["IPONDI"].astype(float)
m = m[~m["TRANS"].astype(str).isin(code_aucun)]           # sans les actifs qui travaillent à domicile
m["voiture"] = m["TRANS"].astype(str).isin(code_voit)
m = m.merge(corr, left_on="COMMUNE", right_on="commune")
g = m.groupby("zone").apply(lambda x: pd.Series({"actifs_mobpro": x["IPONDI"].sum(),
        "part_voiture_mobpro": np.average(x["voiture"], weights=x["IPONDI"])}), include_groups=False)
g.loc["Ensemble"] = [m["IPONDI"].sum(), np.average(m["voiture"], weights=m["IPONDI"])]
display(g.round(3))

## 7. Les tableaux EMC² sur le mode et le motif
On cherche les onglets qui parlent à la fois du **travail** et de la **voiture**, et on les affiche.
Repérez celui qui donne, **pour le motif travail**, la part de la voiture (conducteur + passager) par zone.

In [ ]:
res = []
for p in fichiers:
    if p.suffix.lower() not in (".xlsx", ".xls"):
        continue
    try:
        for sh, raw in pd.read_excel(p, sheet_name=None, header=None, dtype=str).items():
            txt = " ".join(raw.fillna("").astype(str).values.ravel()).lower()
            sc = ("travail" in txt) + ("voiture" in txt or " vp" in txt or "vp " in txt) \
                 + ("mode" in txt) + ("motif" in txt) + (zcol is not None and zcol.lower() in txt)
            if sc >= 3:
                titre = next((c for c in raw.fillna("").astype(str).values.ravel() if len(c) > 15), "")
                res.append((sc, p, sh, titre[:110]))
    except Exception:
        pass
res.sort(key=lambda t: -t[0])
for i, (sc, p, sh, titre) in enumerate(res[:25]):
    print(f"[{i}] {p.name} › {sh} — {titre}")

In [ ]:
CHOIX = 0          # numéro du tableau à afficher (liste ci-dessus)
if res:
    _, p, sh, _ = res[CHOIX]
    print(p.name, "›", sh)
    display(pd.read_excel(p, sheet_name=sh, header=None).dropna(how="all").dropna(axis=1, how="all").head(80))

## 8. Recopier les parts EMC² et comparer
Dans le tableau affiché, lisez **pour le motif travail** la part des déplacements faits en voiture
(conducteur + passager), zone par zone, et recopiez-la ci-dessous en % (ex. `"01": 72.5`).
Les noms de zones doivent être ceux du tableau de la section 6. Laissez `{}` pour ne comparer que l'ensemble.

In [ ]:
EMC2_PART_VOITURE_TRAVAIL = {
    # "01": 72.5,
    # "Ensemble": 80.1,
}

if not EMC2_PART_VOITURE_TRAVAIL:
    print("Recopiez d'abord les valeurs EMC² ci-dessus, puis relancez cette cellule.")
else:
    e = pd.Series(EMC2_PART_VOITURE_TRAVAIL, name="part_voiture_emc2") / 100
    comp = g.join(e, how="inner")
    comp["ecart_points"] = (comp["part_voiture_mobpro"] - comp["part_voiture_emc2"]) * 100
    display(comp.assign(part_voiture_mobpro=lambda x: (x["part_voiture_mobpro"] * 100).round(1),
                        part_voiture_emc2=lambda x: (x["part_voiture_emc2"] * 100).round(1)).round(1))
    ecart = comp["ecart_points"].abs()
    print(f"Écart moyen : {ecart.mean():.1f} points ; écart maximal : {ecart.max():.1f} points ({ecart.idxmax()})")
    print("Lecture : moins de 5 points d'écart = sources cohérentes ; plus de 10 points = à examiner "
          "(zone trop petite dans l'enquête, mode habituel ≠ mode du jour, télétravail…).")
    ax = comp[["part_voiture_mobpro", "part_voiture_emc2"]].mul(100).plot.barh(figsize=(8, 0.4 * len(comp) + 1.5))
    ax.set_xlabel("% voiture pour aller travailler"); ax.legend(["Recensement (MOBPRO)", "Enquête (EMC²)"])
    comp.to_csv(f"controle_emc2_{EMC2_DATASET}.csv")